<a href="https://colab.research.google.com/github/rahuldevagiri/deep-learning-project/blob/main/notebooks/03_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 5.1: setup

In [2]:
!pip install -q segmentation-models-pytorch
import os, json, glob, random, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from matplotlib.colors import ListedColormap
from google.colab import drive
from tqdm.auto import tqdm
drive.mount("/content/drive")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

Mounted at /content/drive
Device: cuda Tesla T4


Cell 5.2b: run this every session. It replaces the old 5.2.

In [3]:
SRC, DATA = "/content/drive/MyDrive/lulc_s1_global", "/content/data"
os.makedirs(DATA, exist_ok=True)

if len(glob.glob(f"{DATA}/patches/*/*.npz")) < 1740:
    !rm -rf "{DATA}/patches"
    !cp "{SRC}/patches.zip" /content/patches.zip
    !unzip -q -o /content/patches.zip -d "{DATA}/"
    !rm /content/patches.zip
!cp "{SRC}/index.csv" "{SRC}/norm_stats.json" "{DATA}/"

n_local = len(glob.glob(f"{DATA}/patches/*/*.npz"))
idx = pd.read_csv(f"{DATA}/index.csv")
idx["path"] = idx.path.str.replace(SRC, DATA, regex=False)
missing = (~idx.path.map(os.path.exists)).sum()
stats = json.load(open(f"{DATA}/norm_stats.json"))
CLASSES, COLORS = stats["classes"], stats["colors"]
NC = len(CLASSES)

print("local files:", n_local, "| index rows:", len(idx), "| missing:", missing)
print(idx.split.value_counts())
assert n_local == 1740 and missing == 0, "Copy incomplete - rerun this cell"

local files: 1740 | index rows: 20575 | missing: 0
split
train    14827
val       3070
test      2678
Name: count, dtype: int64


Cell M2.0: apply the leakage exclusion (between 5.2b and 5.3)

In [4]:
excl = json.load(open(f"{SRC}/exclude_pids.json"))["leakage_drop_pids"]
idx["pid_file"] = idx.path.str.extract(r"p(\d+)\.npz", expand=False).astype(int)
before = len(idx)
idx = idx[~idx.pid_file.isin(excl)].reset_index(drop=True)
print(f"excluded pids {excl}: removed {before - len(idx)} samples (patch x month)")
print(idx.split.value_counts())

excluded pids [1633]: removed 12 samples (patch x month)
split
train    14815
val       3070
test      2678
Name: count, dtype: int64


Cell 5.3: preprocessing and the Dataset

In [5]:
def preprocess(x2, stats):
    """x2: (2, H, W) int16 dB*100 [VV, VH] -> (3, H, W) float32 normalized, nodata mask."""
    nod = (x2[0] == stats["nodata"]) | (x2[1] == stats["nodata"])
    vv = np.clip(x2[0] / stats["scale"], *stats["clip"]["VV"])
    vh = np.clip(x2[1] / stats["scale"], *stats["clip"]["VH"])
    img = np.stack([vv, vh, vv - vh]).astype(np.float32)
    mean = np.array(stats["mean"], np.float32)[:, None, None]
    std = np.array(stats["std"], np.float32)[:, None, None]
    img = (img - mean) / std
    img[:, nod] = 0.0
    return img, nod

class S1Dataset(Dataset):
    def __init__(self, df, stats, augment=False):
        self.df, self.stats, self.augment = df.reset_index(drop=True), stats, augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        d = np.load(r.path)
        img, nod = preprocess(d["x"][r.month], self.stats)
        y = d["y"].astype(np.int64)
        y[nod] = 255                                   # ignore gaps
        if self.augment:                               # geometry only, no value changes
            k = random.randint(0, 3)
            img, y = np.rot90(img, k, (1, 2)), np.rot90(y, k)
            if random.random() < 0.5:
                img, y = img[:, :, ::-1], y[:, ::-1]
        return torch.from_numpy(img.copy()), torch.from_numpy(y.copy())

ds = {s: S1Dataset(idx[idx.split == s], stats, augment=(s == "train")) for s in ("train", "val", "test")}
BS = 16
dl = {s: DataLoader(ds[s], batch_size=BS, shuffle=(s == "train"), num_workers=2,
                    pin_memory=True, drop_last=(s == "train")) for s in ds}
print({s: len(ds[s]) for s in ds})

{'train': 14815, 'val': 3070, 'test': 2678}


5.6: build the fast cache

In [6]:
CACHE = "/content/cache"; os.makedirs(CACHE, exist_ok=True)
idx = idx.reset_index(drop=True)
paths = sorted(idx.path.unique())
pid = {p: i for i, p in enumerate(paths)}
idx["pid"] = idx.path.map(pid)

if not os.path.exists(f"{CACHE}/done"):
    X = np.lib.format.open_memmap(f"{CACHE}/X.npy", mode="w+", dtype=np.int16,
                                  shape=(len(idx), 2, 256, 256))
    Y = np.zeros((len(paths), 256, 256), np.uint8)
    for p, g in tqdm(idx.groupby("path")):
        d = np.load(p); x = d["x"]
        Y[pid[p]] = d["y"]
        for row, m in zip(g.index, g.month):
            X[row] = x[m]
    X.flush(); del X
    np.save(f"{CACHE}/Y.npy", Y)
    open(f"{CACHE}/done", "w").close()
!ls -lh {CACHE}
assert np.load(f"{CACHE}/X.npy", mmap_mode="r").shape[0] == len(idx), "Stale cache - delete /content/cache and rerun 5.6"

  0%|          | 0/1723 [00:00<?, ?it/s]

total 5.2G
-rw-r--r-- 1 root root    0 Nov  5 03:00 done
-rw-r--r-- 1 root root 5.1G Nov  5 03:00 X.npy
-rw-r--r-- 1 root root 108M Nov  5 03:00 Y.npy


New cell 5.7: fast Dataset and DataLoaders

In [7]:
class S1Fast(Dataset):
    def __init__(self, rows, stats, augment=False):
        self.rows, self.pids = rows.index.to_numpy(), rows.pid.to_numpy()
        self.stats, self.augment, self.X, self.Y = stats, augment, None, None

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        if self.X is None:                                  # open lazily inside each worker
            self.X = np.load(f"{CACHE}/X.npy", mmap_mode="r")
            self.Y = np.load(f"{CACHE}/Y.npy", mmap_mode="r")
        img, nod = preprocess(np.asarray(self.X[self.rows[i]]), self.stats)
        y = self.Y[self.pids[i]].astype(np.int64)
        y[nod] = 255
        if self.augment:
            k = random.randint(0, 3)
            img, y = np.rot90(img, k, (1, 2)), np.rot90(y, k)
            if random.random() < 0.5:
                img, y = img[:, :, ::-1], y[:, ::-1]
        return torch.from_numpy(img.copy()), torch.from_numpy(y.copy())

ds = {s: S1Fast(idx[idx.split == s], stats, augment=(s == "train")) for s in ("train", "val", "test")}
dl = {s: DataLoader(ds[s], batch_size=BS, shuffle=(s == "train"), num_workers=2,
                    pin_memory=True, drop_last=(s == "train"),
                    persistent_workers=True, prefetch_factor=4) for s in ds}

t0 = time.time()
for k, (xb, yb) in enumerate(dl["train"]):
    if k == 20: break
print(f"20 batches: {time.time() - t0:.1f}s  ->  approx {(time.time() - t0) / 20 * len(dl['train']) / 60:.1f} min/epoch data time")

20 batches: 1.2s  ->  approx 1.0 min/epoch data time


Cell M2.1: protocol, model factory, loss

In [8]:
import time, json
CFG = {"epochs": 15, "lr": 3e-4, "weight_decay": 1e-4, "batch_size": BS, "seed": 42,
       "loss": "weighted CE + Dice", "channels": stats["channels"], "classes": CLASSES,
       "split": "5-degree grid cells (v1), pid 1633 excluded", "selection": "val mIoU only"}
print(json.dumps(CFG, indent=2))

MODELS = ["unet_r34", "deeplabv3p_r34", "segformer_b2"]

def build_model(name):
    """Every model: same 3 input channels, same 7 classes, ImageNet-initialised encoder."""
    if name == "unet_r34":
        return smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=NC)
    if name == "deeplabv3p_r34":
        return smp.DeepLabV3Plus("resnet34", encoder_weights="imagenet", in_channels=3, classes=NC)
    if name == "segformer_b2":
        return smp.Segformer("mit_b2", encoder_weights="imagenet", in_channels=3, classes=NC)
    raise ValueError(name)

w = torch.tensor(stats["class_weights"], dtype=torch.float32, device=DEVICE)
ce = nn.CrossEntropyLoss(weight=w, ignore_index=255)
dice = smp.losses.DiceLoss(mode="multiclass", ignore_index=255)
def loss_fn(logits, y):
    return ce(logits, y) + dice(logits, y)

{
  "epochs": 15,
  "lr": 0.0003,
  "weight_decay": 0.0001,
  "batch_size": 16,
  "seed": 42,
  "loss": "weighted CE + Dice",
  "channels": [
    "VV",
    "VH",
    "VV-VH"
  ],
  "classes": [
    "Tree",
    "Shrub/Grass",
    "Cropland",
    "Built-up",
    "Bare",
    "Water",
    "Wetland"
  ],
  "split": "5-degree grid cells (v1), pid 1633 excluded",
  "selection": "val mIoU only"
}


Cell M2.2: profiling (about 3–5 min)

In [14]:
def profile(name, bs=BS, iters=10):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = build_model(name).to(DEVICE).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    x = torch.randn(bs, 3, 256, 256, device=DEVICE).to(memory_format=torch.channels_last)
    y = torch.randint(0, NC, (bs, 256, 256), device=DEVICE)

    def step():
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x)
        loss = loss_fn(out.float(), y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        return out

    for _ in range(3): out = step()                  # warm-up (cuDNN picks algorithms)
    torch.cuda.synchronize(); t0 = time.time()
    for _ in range(iters): step()
    torch.cuda.synchronize(); ms = (time.time() - t0) / iters * 1000

    model.eval()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        torch.cuda.synchronize(); t1 = time.time()
        for _ in range(iters): model(x)
        torch.cuda.synchronize(); infer_ms = (time.time() - t1) / iters / bs * 1000

    res = {"model": name, "params_M": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
           "out_shape": tuple(out.shape), "train_ms_per_batch": round(ms),
           "infer_ms_per_patch": round(infer_ms, 1),
           "peak_mem_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
           "gpu_only_min_per_epoch": round(ms * len(dl["train"]) / 60000, 1)}
    del model, opt; torch.cuda.empty_cache()
    return res

prof = pd.DataFrame([profile(m) for m in MODELS])
prof.to_csv(f"{SRC}/models_compare_profile.csv", index=False)
prof

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 98.9MB            

model.safetensors: downloading bytes:           |  0.00B            

,model,params_M,out_shape,train_ms_per_batch,infer_ms_per_patch,peak_mem_GB,gpu_only_min_per_epoch
0,unet_r34,24.4,"(16, 7, 256, 256)",105,1.8,1.56,1.6
1,deeplabv3p_r34,22.4,"(16, 7, 256, 256)",80,1.4,1.35,1.2
2,segformer_b2,24.7,"(16, 7, 256, 256)",145,3.0,2.95,2.2


Cell B0.1: shared evaluation helpers. These are reused for all models in M2.4.

In [9]:
import os
from scipy.ndimage import uniform_filter
from sklearn.ensemble import RandomForestClassifier

COMP = f"{SRC}/compare"; os.makedirs(COMP, exist_ok=True)
Xmm = np.load(f"{CACHE}/X.npy", mmap_mode="r")      # S1 cache (rows = idx rows)
Ymm = np.load(f"{CACHE}/Y.npy", mmap_mode="r")      # labels (rows = idx.pid)

def cm_from(y, p):
    """Confusion matrix: rows = true class, columns = predicted class."""
    return np.bincount(y * NC + p, minlength=NC * NC).reshape(NC, NC)

def metrics_np(cm):
    """Per-class and macro metrics. Classes with no reference pixels -> NaN, excluded from macro means."""
    cm = cm.astype(float); tp = np.diag(cm)
    ref, pred = cm.sum(1), cm.sum(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        prec = np.where(pred > 0, tp / pred, np.nan)
        rec = np.where(ref > 0, tp / ref, np.nan)
        f1 = 2 * prec * rec / (prec + rec)
        iou = np.where(ref > 0, tp / (ref + pred - tp), np.nan)
    return {"mIoU": np.nanmean(iou), "pixel_acc": tp.sum() / cm.sum(), "macro_F1": np.nanmean(f1),
            "IoU": iou, "F1": f1, "precision": prec, "recall": rec}

Cell B0.2: pixel features and training sample

In [16]:
def pixel_features(row):
    """7 features per pixel: VV, VH, VV-VH (normalized) + 7x7 mean and std of VV and VH."""
    img, nod = preprocess(np.asarray(Xmm[row]), stats)           # (3, 256, 256)
    feats = [img[0], img[1], img[2]]
    for c in (0, 1):                                              # VV, VH
        m = uniform_filter(img[c], size=7)                        # local mean
        m2 = uniform_filter(img[c] ** 2, size=7)
        feats += [m, np.sqrt(np.clip(m2 - m ** 2, 0, None))]      # local std = sqrt(E[x^2] - E[x]^2)
    return np.stack(feats, axis=-1), nod                          # (256, 256, 7)

def sample_pixels(rows, n_px, seed):
    rng = np.random.default_rng(seed)
    Xs, ys, rs = [], [], []
    for r in tqdm(rows):
        F, nod = pixel_features(r)
        y = Ymm[idx.pid.iat[r]].astype(np.int64)
        cand = np.flatnonzero(((~nod) & (y != 255)).ravel())     # valid, labelled pixels
        if len(cand) == 0:
            continue
        pick = rng.choice(cand, min(n_px, len(cand)), replace=False)
        Xs.append(F.reshape(-1, 7)[pick]); ys.append(y.ravel()[pick]); rs.append(np.full(len(pick), r))
    return np.concatenate(Xs).astype(np.float32), np.concatenate(ys), np.concatenate(rs)

train_rows = idx.index[idx.split == "train"].to_numpy()
rng = np.random.default_rng(42)
sub_rows = rng.choice(train_rows, 3000, replace=False)           # 3,000 of 14,815 train samples
Xtr, ytr, _ = sample_pixels(sub_rows, n_px=100, seed=42)         # ~300k pixels
print("train pixels:", Xtr.shape, "| class counts:", np.bincount(ytr, minlength=NC))

  0%|          | 0/3000 [00:00<?, ?it/s]

train pixels: (300000, 7) | class counts: [61295 79943 39593 29589 32337 41259 15984]


Cell B0.3: train both forests

In [17]:
cw = {i: wgt for i, wgt in enumerate(stats["class_weights"])}    # same class weights as the CNNs
VARIANTS = {"B0a_rf_pixel": [0, 1, 2], "B0b_rf_context": [0, 1, 2, 3, 4, 5, 6]}
forests, fit_time = {}, {}
for name, cols in VARIANTS.items():
    rf = RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=5,
                                class_weight=cw, n_jobs=-1, random_state=42)
    t0 = time.time(); rf.fit(Xtr[:, cols], ytr); fit_time[name] = time.time() - t0
    forests[name] = rf
    print(f"{name}: fitted in {fit_time[name]:.0f}s")

B0a_rf_pixel: fitted in 35s
B0b_rf_context: fitted in 89s


Cell B0.4: evaluate on validation

In [18]:
val_rows = idx.index[idx.split == "val"].to_numpy()
Xva, yva, rva = sample_pixels(val_rows, n_px=1024, seed=7)       # fixed val pixel sample (~3.1M)
print("val pixels:", Xva.shape)

rows_out = []
for name, cols in VARIANTS.items():
    t0 = time.time(); pva = forests[name].predict(Xva[:, cols]); pt = time.time() - t0
    # per-sample confusion matrices (needed later for the grid-cell bootstrap)
    order = np.argsort(rva, kind="stable"); r_s, y_s, p_s = rva[order], yva[order], pva[order]
    uniq, start = np.unique(r_s, return_index=True)
    cms = np.stack([cm_from(y_s[a:b], p_s[a:b]) for a, b in zip(start, list(start[1:]) + [len(r_s)])])
    np.savez_compressed(f"{COMP}/val_cms_{name}.npz", rows=uniq, pid_file=idx.pid_file.values[uniq], cms=cms)

    m = metrics_np(cms.sum(0))
    rows_out.append({"model": name, "val_mIoU": round(m["mIoU"], 3), "pixel_acc": round(m["pixel_acc"], 3),
                     "macro_F1": round(m["macro_F1"], 3), "fit_s": round(fit_time[name]),
                     "infer_ms_per_patch": round(pt / len(pva) * 65536 * 1000, 1),
                     **{f"IoU_{c}": round(v, 3) for c, v in zip(CLASSES, m["IoU"])}})

b0 = pd.DataFrame(rows_out)
b0.to_csv(f"{COMP}/results_B0.csv", index=False)
b0.T

  0%|          | 0/3070 [00:00<?, ?it/s]

val pixels: (3143680, 7)


,0,1
model,B0a_rf_pixel,B0b_rf_context
val_mIoU,0.235,0.294
pixel_acc,0.409,0.477
macro_F1,0.356,0.426
fit_s,35,89
infer_ms_per_patch,1109.4,991.8
IoU_Tree,0.364,0.426
IoU_Shrub/Grass,0.231,0.3
IoU_Cropland,0.133,0.15
IoU_Built-up,0.212,0.385


Cell M2.3a: training helpers

In [10]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def make_loaders(seed):
    """Fresh loaders per model: same shuffle order + same augmentation stream for every model."""
    g = torch.Generator().manual_seed(seed)
    tr = DataLoader(ds["train"], batch_size=BS, shuffle=True, num_workers=2, pin_memory=True,
                    drop_last=True, persistent_workers=True, prefetch_factor=4, generator=g)
    va = DataLoader(ds["val"], batch_size=BS, shuffle=False, num_workers=2, pin_memory=True,
                    persistent_workers=True, prefetch_factor=4)
    return tr, va

def update_cm(cm, logits, y):
    pred, m = logits.argmax(1), y != 255
    cm += torch.bincount(y[m] * NC + pred[m], minlength=NC * NC).reshape(NC, NC)
    return cm

def run_epoch(model, loader, train, opt=None, scaler=None):
    model.train(train)
    cm = torch.zeros(NC, NC, dtype=torch.long, device=DEVICE)
    tot, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in tqdm(loader, leave=False):
            xb = xb.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
            yb = yb.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(xb)
            loss = loss_fn(out.float(), yb)
            if train:
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * xb.size(0); n += xb.size(0)
            cm = update_cm(cm, out.detach(), yb)
    m = metrics_np(cm.cpu().numpy()); m["loss"] = tot / n
    return m

Cell M2.3b: train one model, resumable

In [11]:
def train_model(name, epochs=CFG["epochs"]):
    out_dir = f"{COMP}/{name}"; os.makedirs(out_dir, exist_ok=True)
    LAST, BEST = f"{out_dir}/last.pth", f"{out_dir}/best.pth"
    set_seed(CFG["seed"])
    model = build_model(name).to(DEVICE).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda")
    start, best, hist = 0, -1.0, []
    if os.path.exists(LAST):
        ck = torch.load(LAST, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        start, best, hist = ck["epoch"] + 1, ck["best"], ck["hist"]
    if start >= epochs:
        print(f"{name}: already finished (best val mIoU {best:.3f})"); return pd.DataFrame(hist)

    tr_dl, va_dl = make_loaders(CFG["seed"] + start)        # resumed runs get a new (logged) order
    torch.cuda.reset_peak_memory_stats()
    print(f"== {name}: epochs {start + 1}-{epochs}")
    for ep in range(start, epochs):
        t0 = time.time()
        tr = run_epoch(model, tr_dl, True, opt, scaler)
        va = run_epoch(model, va_dl, False)
        sched.step()
        rec = {"model": name, "epoch": ep + 1, "lr": opt.param_groups[0]["lr"], "epoch_s": round(time.time() - t0),
               "train_loss": tr["loss"], "val_loss": va["loss"], "train_mIoU": tr["mIoU"], "val_mIoU": va["mIoU"],
               "val_acc": va["pixel_acc"], "peak_mem_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2),
               "resumed_at": start, **{f"IoU_{c}": float(v) for c, v in zip(CLASSES, va["IoU"])}}
        hist.append(rec)
        is_best = va["mIoU"] > best
        if is_best:
            best = va["mIoU"]
            torch.save({"model": model.state_dict(), "name": name, "cfg": CFG, "stats": stats,
                        "epoch": ep + 1, "val_mIoU": best}, BEST)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "best": best, "hist": hist}, LAST)
        pd.DataFrame(hist).to_csv(f"{out_dir}/history.csv", index=False)
        print(f"{name} ep {ep + 1:02d} | {rec['epoch_s']:4d}s | train loss {tr['loss']:.3f} mIoU {tr['mIoU']:.3f} | "
              f"val loss {va['loss']:.3f} mIoU {va['mIoU']:.3f}" + ("  *best" if is_best else ""))
    del model, opt, tr_dl, va_dl; torch.cuda.empty_cache()
    return pd.DataFrame(hist)

Cell M2.3c: run all three

In [12]:
histories = {name: train_model(name) for name in MODELS}

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

unet_r34: already finished (best val mIoU 0.512)
deeplabv3p_r34: already finished (best val mIoU 0.518)


config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 98.9MB            

model.safetensors: downloading bytes:           |  0.00B            

segformer_b2: already finished (best val mIoU 0.527)


Cell M2.3d: final validation with per-sample confusion matrices

In [13]:
def eval_val_per_sample(name):
    ck = torch.load(f"{COMP}/{name}/best.pth", map_location=DEVICE, weights_only=False)
    model = build_model(name).to(DEVICE).to(memory_format=torch.channels_last)
    model.load_state_dict(ck["model"]); model.eval()
    loader = DataLoader(ds["val"], batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
    cms = []
    with torch.no_grad():
        for xb, yb in tqdm(loader, leave=False):
            xb = xb.to(DEVICE, memory_format=torch.channels_last); yb = yb.to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                pred = model(xb).argmax(1)
            for k in range(len(yb)):
                m = yb[k] != 255
                cms.append(torch.bincount(yb[k][m] * NC + pred[k][m], minlength=NC * NC)
                           .reshape(NC, NC).cpu().numpy())
    cms = np.stack(cms)
    rows = ds["val"].rows                                   # same order as the loader
    np.savez_compressed(f"{COMP}/val_cms_{name}.npz", rows=rows, pid_file=idx.pid_file.values[rows], cms=cms)
    m = metrics_np(cms.sum(0))
    h = histories[name] if name in histories else pd.read_csv(f"{COMP}/{name}/history.csv")
    return {"model": name, "best_epoch": ck["epoch"], "val_mIoU": round(m["mIoU"], 3),
            "pixel_acc": round(m["pixel_acc"], 3), "macro_F1": round(m["macro_F1"], 3),
            "train_min_total": round(h.epoch_s.sum() / 60, 1),
            **{f"IoU_{c}": round(v, 3) for c, v in zip(CLASSES, m["IoU"])}}

dl_res = pd.DataFrame([eval_val_per_sample(n) for n in MODELS])
dl_res.to_csv(f"{COMP}/results_M2_dl.csv", index=False)
pd.concat([pd.read_csv(f"{COMP}/results_B0.csv"), dl_res], ignore_index=True).T

  0%|          | 0/192 [00:00<?, ?it/s]

  0%|          | 0/192 [00:00<?, ?it/s]

  0%|          | 0/192 [00:00<?, ?it/s]

,0,1,2,3,4
model,B0a_rf_pixel,B0b_rf_context,unet_r34,deeplabv3p_r34,segformer_b2
val_mIoU,0.235,0.294,0.512,0.518,0.527
pixel_acc,0.409,0.477,0.668,0.672,0.686
macro_F1,0.356,0.426,0.663,0.67,0.676
fit_s,35.0,89.0,NaN,NaN,NaN
infer_ms_per_patch,1109.4,991.8,NaN,NaN,NaN
IoU_Tree,0.364,0.426,0.549,0.529,0.569
IoU_Shrub/Grass,0.231,0.3,0.307,0.318,0.348
IoU_Cropland,0.133,0.15,0.561,0.587,0.607
IoU_Built-up,0.212,0.385,0.608,0.616,0.636
